# 📝 과제 01: 온라인 쇼핑 상담 서버의 지연, 처리량, 비용 측정하기

## 만들 결과

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01(서빙 성능 벤치마킹)의 측정 순서를 온라인 쇼핑 상담 창구(전자상거래법 청약철회, 환급·반환 비용)에 그대로 적용합니다. 내 PC에서 참고 조문이 든 시스템 메시지로 벤치마크 입력 파일을 만들어 RunPod에 올리고, RunPod 터미널에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>로 동시 요청 수를 바꿔 가며 지연과 처리량을 측정합니다. 내려받은 결과 JSON으로 지연 목표를 지키는 동시 요청 수와 100만 토큰당 비용을 계산합니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/20_gwaje01_pipeline.png" width="1050" alt="과제 01의 다섯 단계: 입력 파일, 벤치마크와 요약 테이블, 동시 요청 수 비교, 동시 요청 수 고르기, 토큰당 비용">

<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">만들 것</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">참고 조문 19개를 붙인 시스템 메시지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_system_prompt</code>와 질문 107개의 벤치마크 입력 파일 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_prompts_path</code></li>
<li style="margin:3px 0;">동시 요청 16, 요청 64개의 벤치마크 결과 파일 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_load_c16.json</code>(지표별 평균과 p50, p95, p99는 RunPod 화면의 요약 테이블에서 읽습니다)</li>
<li style="margin:3px 0;">동시 요청 수 1, 16, 64의 결과 JSON 딕셔너리 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_results</code>와 출력 처리량과 요청 처리량, p95 지연, goodput, 목표 달성률 테이블 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_table</code></li>
<li style="margin:3px 0;">지연 목표(TTFT 1000ms, TPOT 50ms)를 95% 이상 지키면서 출력 처리량이 가장 큰 동시 요청 수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_concurrency</code></li>
<li style="margin:3px 0;">100만 토큰당 비용 열을 더한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cost_table</code>과 동시 요청 1의 비용이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_concurrency</code> 비용의 몇 배인지 나타낸 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cost_ratio</code></li>
</ul>
</div>

## 준비: 데이터와 제공 코드

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">환경:</strong> 서버와 벤치마크는 RunPod 터미널에서, 이 노트북은 내 PC에서 실행합니다. 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에는 교안 01과 같은 서버 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_SERVER_URL</code>과 API 키 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>가 있어야 합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자료:</strong> 참고 조문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/과제/commerce_reference.json</code>에 있습니다(전자상거래법 청약철회, 환급·반환 비용 조문 19개). 모양은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;title&quot;: ..., &quot;articles&quot;: [{&quot;doc_id&quot;: ..., &quot;text&quot;: ...}, ...]}</code>입니다. 질문 107개는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/과제/commerce_questions.jsonl</code>에 있고, 행마다 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">id</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">question</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">source</code>가 있습니다. 제공 코드가 두 파일을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_reference</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_questions</code>로 읽어 둡니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">제공 코드:</strong> 경로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_dir</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_dir</code>(RunPod에서 내려받은 결과 JSON을 두는 폴더), 서버 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVER_URL</code>, 상수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">MODEL_NAME</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVED_NAME</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">MAX_TOKENS</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">TTFT_TARGET_MS</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">TPOT_TARGET_MS</code>, 함수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_jsonl(path)</code>, 교안 01과 같은 함수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">build_prompts(questions, system_prompt, path)</code>, 지침 문장 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SYSTEM_PROMPT_COMMERCE</code>, 토크나이저 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tokenizer</code></li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 01과 다른 점:</strong> 지침 문장이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SYSTEM_PROMPT_COMMERCE</code>이고 참고 조문과 질문이 온라인 쇼핑 것이며, 결과 이름이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_</code>로 시작합니다. 서버, 벤치마크 옵션(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>), 지연 목표(TTFT 1000ms, TPOT 50ms)는 교안 01과 같습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성:</strong> 코드 셀은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO</code> 주석 아래의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>를 채우고 바로 아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[자가채점]</code> 셀을 실행합니다. RunPod 터미널 명령은 블록의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;...&gt;</code> 칸을 요구사항의 값으로 바꿔 실행합니다. 뒤 문항이 앞 문항의 결과를 쓰므로 1번부터 차례로 풉니다.</li>
</ul>
</div>

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from transformers import AutoTokenizer

# 내 PC의 .env에서 서버의 공개 주소와 API 키를 읽습니다
load_dotenv(find_dotenv(usecwd=True))
material_dir = Path(".").resolve()  # 이 단원 폴더(정답 노트북은 "..")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
bench_dir = output_dir / "bench"           # RunPod에서 내려받은 벤치마크 결과 JSON을 두는 폴더
sweep_dir = output_dir / "sweep"           # RunPod에서 내려받은 설정 비교 결과(summary.csv)를 두는 폴더
bench_dir.mkdir(parents=True, exist_ok=True)
sweep_dir.mkdir(parents=True, exist_ok=True)
SERVER_URL = os.environ["VLLM_SERVER_URL"].rstrip("/")  # RunPod 공개 HTTPS 주소
VLLM_API_KEY = os.environ["VLLM_API_KEY"]

# 설정값
MODEL_NAME = "Qwen/Qwen3.5-4B"            # 서버가 불러온 허브 모델(토크나이저도 이 이름으로 받습니다)
SERVED_NAME = "qwen3.5-4b"               # 서버 이름(--served-model-name). 요청의 model에 넣습니다
MAX_MODEL_LEN = 8192                  # 참고 조문이 든 시스템 메시지 + 질문 + 답을 담는 최대 길이
MAX_TOKENS = 256                     # 요청마다 생성 상한(지난 단원과 같은 값)
TTFT_TARGET_MS = 1000                # 지연 목표: 첫 토큰까지 1000ms 안
TPOT_TARGET_MS = 50                  # 지연 목표: 출력 토큰 하나에 50ms 안

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
import platform

import matplotlib.pyplot as plt
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스: 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 온라인 쇼핑 상담 창구의 지침 문장. 시스템 메시지는 이 문장 뒤에 참고 조문을 붙여 만듭니다
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."

def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# 온라인 쇼핑 상담 창구의 참고 조문(제목과 조문 리스트)과 질문
with open(data_dir / "과제" / "commerce_reference.json", encoding="utf-8") as f:
    commerce_reference = json.load(f)
commerce_questions = load_jsonl(data_dir / "과제" / "commerce_questions.jsonl")
print("참고 조문:", commerce_reference["title"], "| 조문 수:", len(commerce_reference["articles"]), "| 질문 수:", len(commerce_questions))

# 서버와 같은 모델의 토크나이저(입력 파일을 만들 때 chat template을 적용합니다)
# HF_TOKEN 없이 내려받으면 'unauthenticated requests' 경고가 찍힙니다. 공개 모델이라 그대로 진행해도 됩니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">build_prompts</code>는 교안 01과 같습니다. 질문마다 chat template을 적용한 프롬프트를 벤치마크 입력 파일로 저장합니다.</p>
</div>

In [ ]:
# 교안 01과 같은 입력 파일 함수: 질문마다 chat template을 적용한 프롬프트를 한 줄씩 저장합니다
def build_prompts(questions, system_prompt, path):
    """질문마다 시스템 메시지와 함께 chat template을 적용한 프롬프트를 vllm bench serve 입력 파일(JSONL)로 저장합니다."""
    with open(path, "w", encoding="utf-8") as f:
        for row in questions:
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": row["question"]}]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            f.write(json.dumps({"prompt": prompt}, ensure_ascii=False) + "\n")  # 벤치마크 도구가 읽는 키는 prompt
    print("입력 파일:", path.name, "| 요청 수:", len(questions))

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버:</strong> 교안 01의 서버가 떠 있으면 그대로 씁니다. 모델과 서버 옵션은 같고, 시스템 메시지만 교안 01과 다릅니다. 서버가 없을 때만 RunPod 터미널 1에서 아래 두 블록을 차례로 실행합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 1**

```bash
# 실습자료 폴더에서 연 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 서버 API 키를 입력합니다. VLLM_API_KEY는 서버가 요청을 인증할 키, OPENAI_API_KEY는 vllm bench serve가 요청에 붙이는 키입니다
read -s -p "서버 API 키 입력: " VLLM_API_KEY && echo && export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY"
```

**복사해서 실행 · RunPod 터미널 1**

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --max-model-len 8192 \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-prompt-tokens-details \
  --enable-prefix-caching \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b_server.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. 서버 출력은 RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/qwen3.5-4b_server.log</code>에도 남습니다. <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 노트북</strong>으로 돌아옵니다.</p>
<p style="margin:6px 0;">첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">벤치마크:</strong> RunPod 터미널 2에서 교안 01에서 정의한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>를 그대로 씁니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 2**

```bash
# 새 터미널이면 이 블록부터 실행합니다
# 실습자료 폴더에서 연 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 서버 API 키를 입력합니다. VLLM_API_KEY는 서버가 요청을 인증할 키, OPENAI_API_KEY는 vllm bench serve가 요청에 붙이는 키입니다
read -s -p "서버 API 키 입력: " VLLM_API_KEY && echo && export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY"
```

**복사해서 실행 · RunPod 터미널 2**

```bash
# 사용법: run_bench <결과 이름> <입력 파일> <동시 요청 수> <요청 수>
# 같은 파드의 서버(127.0.0.1:8000)에 요청을 보내고, 결과 JSON은 output/bench/<결과 이름>.json에,
# 화면의 요약 테이블은 output/bench/<결과 이름>.txt에도 남깁니다
run_bench() {
  uv run vllm bench serve \
    --backend openai \
    --base-url http://127.0.0.1:8000 \
    --model qwen3.5-4b \
    --tokenizer Qwen/Qwen3.5-4B \
    --dataset-name custom \
    --dataset-path "$2" \
    --skip-chat-template \
    --custom-output-len 256 \
    --num-prompts "$4" \
    --max-concurrency "$3" \
    --temperature 0 \
    --seed 0 \
    --percentile-metrics ttft,tpot,itl,e2el \
    --metric-percentiles 50,95,99 \
    --goodput ttft:1000 tpot:50 \
    --save-result --save-detailed \
    --result-dir output/bench \
    --result-filename "$1.json" | tee "output/bench/$1.txt"
}
```

터미널 2를 새로 열면 준비 블록과 이 블록을 다시 실행합니다.

## 1. 시스템 메시지와 입력 파일

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 상담 서버가 실제로 받는 요청(같은 시스템 메시지 + 질문)으로 부하를 걸기 위해, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>가 읽을 입력 파일을 만듭니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">commerce_articles</code></strong>(list): <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_reference[&quot;articles&quot;]</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">text</code>를 조문 순서대로 모은 리스트</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">commerce_system_prompt</code></strong>(str): <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SYSTEM_PROMPT_COMMERCE + &quot;\n\n[참고 조문]\n&quot;</code> 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_articles</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;\n&quot;.join</code>으로 이은 문자열을 붙인 것(마지막 조문 뒤에는 줄바꿈이 없습니다)</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">commerce_prompts_path</code></strong>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_dir / &quot;commerce_prompts.jsonl&quot;</code>. 제공 함수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">build_prompts</code>로 질문 전체 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_questions</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_system_prompt</code>를 이 경로에 저장합니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 2-2(시스템 메시지)와 4-1(입력 파일)을 참고합니다.
조문 리스트는 articles의 항목마다 text 값을 꺼내 순서대로 모읍니다.
build_prompts에는 질문 리스트, 시스템 메시지, 저장 경로를 이 순서로 넘깁니다.
```

</details>

In [ ]:
# TODO (1): 참고 조문의 text를 조문 순서대로 모은 리스트를 만드세요
commerce_articles = ...

# TODO (2): 시스템 메시지 = 지침 문장 + 빈 줄 + [참고 조문] + 조문들(한 줄에 조문 하나)
commerce_system_prompt = ...
print("시스템 메시지 토큰 수:", len(tokenizer.encode(commerce_system_prompt)))

# TODO (3): 입력 파일 경로를 정하고, 다음 줄에서 build_prompts로 질문 전체를 저장하세요
commerce_prompts_path = ...
...

In [ ]:
# [자가채점] 시스템 메시지의 모양과 조문 순서, 입력 파일의 줄 수와 프롬프트 모양을 확인합니다
assert isinstance(commerce_articles, list) and len(commerce_articles) == len(commerce_reference["articles"]), f"commerce_articles는 조문 {len(commerce_reference['articles'])}개의 text를 담은 리스트입니다."
assert commerce_articles[0] == commerce_reference["articles"][0]["text"], "commerce_articles의 첫 항목은 첫 조문의 text 문자열입니다(doc_id나 딕셔너리가 아닙니다)."

assert isinstance(commerce_system_prompt, str), "commerce_system_prompt는 문자열입니다. `...`를 채웠는지 확인하세요."
assert commerce_system_prompt.startswith(SYSTEM_PROMPT_COMMERCE + "\n\n[참고 조문]\n"), "commerce_system_prompt는 SYSTEM_PROMPT_COMMERCE, 빈 줄, \"[참고 조문]\" 머리와 줄바꿈으로 시작합니다."

# 조문이 하나도 빠지지 않고 조문 순서대로 들어 있는지 봅니다
position = 0
for article in commerce_reference["articles"]:
    found_at = commerce_system_prompt.find(article["text"], position)
    assert found_at >= 0, f"{article['doc_id']} 조문이 앞 조문 뒤에 없습니다. 조문 리스트를 조문 순서대로 모두 이었는지 확인하세요."
    position = found_at + len(article["text"])
assert commerce_system_prompt.endswith("\n".join(commerce_articles)), "조문은 \"\\n\".join으로 한 줄에 하나씩 잇고, 마지막 조문 뒤에는 아무것도 붙이지 않습니다."
assert commerce_system_prompt == SYSTEM_PROMPT_COMMERCE + "\n\n[참고 조문]\n" + "\n".join(commerce_articles), "시스템 메시지가 요구사항 (2)의 모양과 다릅니다. 머리와 조문 사이, 조문과 조문 사이에 줄바꿈이 하나씩인지, 조문을 한 번씩만 이었는지 확인하세요."

# 입력 파일: 질문마다 한 줄, 줄마다 같은 시스템 메시지
assert commerce_prompts_path == output_dir / "commerce_prompts.jsonl", f"commerce_prompts_path가 {commerce_prompts_path}입니다. output_dir / \"commerce_prompts.jsonl\"로 두세요."
prompt_rows = load_jsonl(commerce_prompts_path)
assert len(prompt_rows) == len(commerce_questions), f"입력 파일이 {len(prompt_rows)}줄입니다. build_prompts에 질문 전체 commerce_questions({len(commerce_questions)}개)를 넘기세요."
for row in prompt_rows:
    assert row["prompt"].startswith("<|im_start|>system"), "프롬프트가 시스템 메시지로 시작하지 않습니다. build_prompts에 시스템 메시지를 둘째 인자로 넘겼는지 확인하세요."
    assert commerce_system_prompt in row["prompt"], "프롬프트에 commerce_system_prompt 전체가 들어 있지 않습니다. build_prompts에 commerce_system_prompt를 넘기세요."

# 첫 줄: 조문이 앞, 질문이 뒤, 끝은 빈 <think></think>(생각 과정을 끈 표시) 뒤의 assistant 차례
first_prompt = prompt_rows[0]["prompt"]
last_article_at = first_prompt.find(commerce_reference["articles"][-1]["text"])
question_at = first_prompt.find(commerce_questions[0]["question"])
assert question_at > last_article_at, "첫 줄에서 질문이 조문보다 뒤에 있지 않습니다. 질문은 user 메시지로 들어가야 합니다."
assert first_prompt.rstrip().endswith("<think>\n\n</think>"), "입력 끝이 빈 <think></think>가 아닙니다. 제공 함수 build_prompts를 그대로 썼는지 확인하세요."
print("✅ 통과!")

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">파일 올리기 · 내 PC → RunPod</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">내 PC 위치</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">RunPod 위치</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_prompts.jsonl</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/commerce_prompts.jsonl</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output/commerce_prompts.jsonl</code></td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">RunPod JupyterLab 왼쪽 파일 목록에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output</code> 폴더를 열고, 업로드 버튼으로 내 PC의 위 파일을 올립니다.</p>
</div>

## 2. 벤치마크 하나와 요약 테이블

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 여러 사용자가 동시에 물을 때 요청마다 지연이 얼마나 다른지 보려고, 벤치마크를 한 번 실행해 RunPod 화면의 요약 테이블에서 지표별 평균과 백분위를 읽습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) RunPod 터미널 2에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>를 결과 이름 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_load_c16</code>, 입력 파일 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/commerce_prompts.jsonl</code>, 동시 요청 수 16, 요청 수 64로 실행합니다. 3번이 이 결과를 다시 씁니다.</li>
</ul>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성할 것:</strong> (1) RunPod 터미널 2에서 실행할 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code> 한 줄을 작성합니다. 형식은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench &lt;결과 이름&gt; &lt;입력 파일&gt; &lt;동시 요청 수&gt; &lt;요청 수&gt;</code>이고, 입력 파일은 RunPod 단원 폴더 기준 경로입니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 4-2(벤치마크 실행)와 4-3(요약 테이블 읽기)을 참고합니다.
교안 01의 run_bench 한 줄에서 결과 이름과 입력 파일만 이 과제의 것으로 바꿉니다.
동시 요청 수와 요청 수는 요구사항 (1)의 값을 순서대로 씁니다.
```

</details>

**복사해서 실행 · RunPod 터미널 2**

```bash
# TODO (1): <...> 네 자리를 요구사항 (1)의 결과 이름, 입력 파일, 동시 요청 수, 요청 수로 바꿔 실행하세요
run_bench <결과 이름> <입력 파일> <동시 요청 수> <요청 수>
```

**RunPod 터미널 2 · 화면에서 확인**

```bash
# 화면에 찍힌 요약 테이블을 다시 봅니다(run_bench가 output/bench/<결과 이름>.txt에도 남깁니다)
cat output/bench/commerce_load_c16.txt
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Successful requests</code>가 64</strong>이고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Maximum request concurrency</code>가 16이면 3번으로 넘어갑니다. 64보다 작으면 RunPod 터미널 1의 서버 화면에 오류가 있는지 봅니다. 지표 이름의 E2EL은 요청 지연입니다.</p>
</div>

## 3. 동시 요청 수 비교

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 서버 한 대에 동시 요청을 몇 개까지 보낼지 정하기 위해, 같은 입력 파일로 동시 요청 수만 1, 16, 64로 바꿔 측정합니다. 동시 요청 16은 2번의 결과를 그대로 쓰므로 새로 돌리는 벤치마크는 두 번입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) RunPod 터미널 2에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>를 두 줄 실행합니다. 동시 요청 1은 요청 8개, 동시 요청 64는 요청 64개를 보냅니다(동시 요청 1은 하나씩 처리해 오래 걸리므로 적게 보냅니다). 결과 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_load_c&lt;동시 요청 수&gt;</code>(예: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_load_c64</code>)이고, 입력 파일은 2번과 같습니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">load_results</code></strong>(dict): 동시 요청 수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">1</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">16</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">64</code>를 키로, 내려받은 결과 JSON <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_dir / &quot;commerce_load_c&lt;동시 요청 수&gt;.json&quot;</code>을 읽은 딕셔너리를 담습니다(16은 2번에서 만든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_load_c16.json</code>).</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">load_table</code></strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pd.DataFrame</code>): 동시 요청 수가 작은 순서로 세 행, 아래 열 순서의 테이블. 목표 달성률은 반올림 전 결과 JSON 값으로 나누고, 테이블 전체를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.round(2)</code>로 반올림합니다.</li>
</ul>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">열</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">값(결과 JSON 키)</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;동시 요청 수&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">동시 요청 수</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;출력 처리량(토큰/초)&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_throughput</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;요청 처리량(요청/초)&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_throughput</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;p95 TTFT(ms)&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">p95_ttft_ms</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;p95 TPOT(ms)&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">p95_tpot_ms</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;goodput(요청/초)&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_goodput</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;목표 달성률&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_goodput / request_throughput</code></td>
</tr>
</tbody>
</table>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성할 것:</strong> (1) RunPod 터미널 2에서 실행할 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code> 두 줄을 작성합니다. 동시 요청 수가 작은 순서로 한 줄에 하나씩 씁니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 5-1을 참고합니다.
2번의 run_bench 한 줄에서 결과 이름의 숫자, 동시 요청 수, 요청 수만 바꿔 두 줄을 씁니다.
동시 요청 1은 하나씩 처리해 오래 걸리므로 요청을 적게 보냅니다.
```

</details>

**복사해서 실행 · RunPod 터미널 2**

```bash
# TODO (1): 줄마다 <...> 네 자리를 요구사항 (1)의 값으로 바꿔 동시 요청 수 1, 64 순서로 실행하세요
run_bench <결과 이름> <입력 파일> <동시 요청 수> <요청 수>
run_bench <결과 이름> <입력 파일> <동시 요청 수> <요청 수>
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">파일 내려받기 · RunPod → 내 PC</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">내 PC 위치</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">RunPod 위치</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_load_c1.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench/commerce_load_c1.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output/bench/commerce_load_c1.json</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_load_c16.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench/commerce_load_c16.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output/bench/commerce_load_c16.json</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_load_c64.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench/commerce_load_c64.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output/bench/commerce_load_c64.json</code></td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">RunPod JupyterLab 왼쪽 파일 목록에서 파일을 오른쪽 클릭해 Download를 누릅니다. 내려받은 파일은 내 PC 위치 열의 폴더(단원 폴더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹</code> 기준)에 저장합니다. 브라우저가 Downloads에 저장했다면 이 위치로 옮깁니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 5-1(결과 JSON 읽기)과 5-3(goodput과 목표 달성률)을 참고합니다.
파일 이름은 동시 요청 수를 넣은 f-string으로 만들고, read_text로 읽은 문자열을 json.loads로 딕셔너리로 바꿉니다.
테이블 칸에는 반올림 전 결과 JSON 값을 넣고, 반올림은 테이블을 만든 뒤 한 번만 합니다.
```

</details>

In [ ]:
# TODO (2): 동시 요청 수(작은 순서)마다 내려받은 결과 JSON을 읽어 딕셔너리에 담으세요
concurrency_levels = ...
load_results = {}
for concurrency in concurrency_levels:
    load_results[concurrency] = ...
    print(f"동시 요청 {concurrency}: 완료 {load_results[concurrency]['completed']}개 / 보낸 요청 {load_results[concurrency]['num_prompts']}개")

# TODO (3): 동시 요청 수마다 결과 JSON 값을 한 행으로 모아 테이블을 만드세요
load_rows = []
for concurrency in concurrency_levels:
    result = load_results[concurrency]
    load_rows.append({
        "동시 요청 수": concurrency,
        "출력 처리량(토큰/초)": ...,
        "요청 처리량(요청/초)": ...,
        "p95 TTFT(ms)": ...,
        "p95 TPOT(ms)": ...,
        "goodput(요청/초)": ...,
        "목표 달성률": ...,  # 두 지연 목표를 모두 지킨 요청의 비율
    })
load_table = ...
display(load_table)

In [ ]:
# [자가채점] 결과 딕셔너리의 키와 파일, 결과 JSON의 입력 파일과 실행 조건(2번의 동시 요청 16 포함), 완료 요청 수, 테이블의 모양과 값, 목표 달성률을 확인합니다
assert isinstance(load_results, dict), "load_results는 동시 요청 수를 키로 한 딕셔너리입니다. `...`를 채웠는지 확인하세요."
assert sorted(load_results.keys()) == [1, 16, 64], f"load_results의 키가 {sorted(load_results.keys())}입니다. 키는 동시 요청 수 1, 16, 64입니다."

# 동시 요청 수마다 결과 파일이 있고, 터미널 명령의 입력 파일, 동시 요청 수, 요청 수가 맞고, 보낸 요청이 모두 끝났는지 봅니다
commerce_input_lens = set()
for row in load_jsonl(commerce_prompts_path):
    commerce_input_lens.add(len(tokenizer(row["prompt"]).input_ids))
sent_prompts = {1: 8, 16: 64, 64: 64}
for concurrency, num_prompts in sent_prompts.items():
    assert (bench_dir / f"commerce_load_c{concurrency}.json").exists(), f"output/bench에 commerce_load_c{concurrency}.json이 없습니다. 결과 이름을 commerce_load_c<동시 요청 수>로 실행하고, 내려받은 파일을 내 PC의 output/bench에 저장하세요."
    result = load_results[concurrency]
    assert result == json.loads((bench_dir / f"commerce_load_c{concurrency}.json").read_text(encoding="utf-8")), f"load_results[{concurrency}]은 commerce_load_c{concurrency}.json을 읽은 딕셔너리입니다. 교안 01의 labor_ 파일을 읽지 않았는지 확인하세요."
    assert set(result["input_lens"]).issubset(commerce_input_lens), f"commerce_load_c{concurrency}의 입력 토큰 수(input_lens)가 commerce_prompts.jsonl의 프롬프트와 다릅니다. run_bench의 둘째 인자를 output/commerce_prompts.jsonl로 실행하세요."
    assert result["max_concurrency"] == concurrency, f"commerce_load_c{concurrency}의 동시 요청 수(max_concurrency)가 {result['max_concurrency']}입니다. 결과 이름의 숫자와 run_bench의 셋째 인자를 같게 실행하세요."
    assert result["num_prompts"] == num_prompts, f"commerce_load_c{concurrency}의 요청 수(num_prompts)가 {result['num_prompts']}입니다. 보낼 요청 수는 {num_prompts}개입니다."
    assert result["completed"] == num_prompts, f"동시 요청 {concurrency}의 완료 요청이 {result['completed']}개입니다. 보낼 요청 수는 {num_prompts}개입니다. RunPod 터미널 1 화면에 오류가 없는지 확인하세요."

expected_columns = ["동시 요청 수", "출력 처리량(토큰/초)", "요청 처리량(요청/초)", "p95 TTFT(ms)", "p95 TPOT(ms)", "goodput(요청/초)", "목표 달성률"]
assert isinstance(load_table, pd.DataFrame), "load_table은 pd.DataFrame으로 만든 테이블입니다."
assert list(load_table.columns) == expected_columns, f"load_table의 열이 {list(load_table.columns)}입니다. 요구사항의 열 순서를 확인하세요."
assert list(load_table["동시 요청 수"]) == [1, 16, 64], f"\"동시 요청 수\" 열이 {list(load_table['동시 요청 수'])}입니다. 동시 요청 수가 작은 순서로 세 행입니다."
assert load_table.equals(load_table.round(2)), "load_table 전체를 .round(2)로 반올림하세요."

# 테이블 값이 결과 JSON에서 왔는지, 목표 달성률이 goodput / 요청 처리량인지 봅니다
table_by_concurrency = load_table.set_index("동시 요청 수")
for concurrency in [1, 16, 64]:
    row = table_by_concurrency.loc[concurrency]
    result = load_results[concurrency]
    assert abs(row["출력 처리량(토큰/초)"] - result["output_throughput"]) <= 0.01, f"동시 요청 {concurrency}의 출력 처리량이 결과 JSON의 output_throughput과 다릅니다."
    assert abs(row["p95 TTFT(ms)"] - result["p95_ttft_ms"]) <= 0.01, f"동시 요청 {concurrency}의 p95 TTFT가 결과 JSON의 p95_ttft_ms와 다릅니다."
    assert abs(row["요청 처리량(요청/초)"] - result["request_throughput"]) <= 0.01, f"동시 요청 {concurrency}의 요청 처리량이 결과 JSON의 request_throughput과 다릅니다."
    assert abs(row["p95 TPOT(ms)"] - result["p95_tpot_ms"]) <= 0.01, f"동시 요청 {concurrency}의 p95 TPOT가 결과 JSON의 p95_tpot_ms와 다릅니다."
    assert abs(row["goodput(요청/초)"] - result["request_goodput"]) <= 0.01, f"동시 요청 {concurrency}의 goodput이 결과 JSON의 request_goodput과 다릅니다."
    assert 0 <= row["목표 달성률"] <= 1, f"동시 요청 {concurrency}의 목표 달성률이 {row['목표 달성률']}입니다. goodput을 요청 처리량으로 나눈 0과 1 사이 값입니다."
    # 반올림 전 결과 JSON 값으로 나눈 목표 달성률과 소수 둘째 자리 반올림 차이 안에서 같아야 합니다
    attainment = result["request_goodput"] / result["request_throughput"]
    assert abs(row["목표 달성률"] - attainment) <= 0.006, f"동시 요청 {concurrency}의 목표 달성률 {row['목표 달성률']}이 결과 JSON 값으로 나눈 {attainment:.2f}와 다릅니다. 반올림 전 request_goodput을 반올림 전 request_throughput으로 나누세요."

# 동시 요청을 늘리면 같은 시간에 만드는 토큰이 늘어납니다
throughput_c1 = table_by_concurrency.loc[1, "출력 처리량(토큰/초)"]
throughput_c64 = table_by_concurrency.loc[64, "출력 처리량(토큰/초)"]
assert throughput_c64 > throughput_c1, f"동시 요청 64의 출력 처리량 {throughput_c64}이 동시 요청 1의 {throughput_c1}보다 크지 않습니다. 결과 이름과 run_bench의 동시 요청 수를 맞춰 실행했는지 확인하세요."
print("✅ 통과!")

## 4. 동시 요청 수 고르기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 지연 목표(TTFT 1000ms, TPOT 50ms)를 지키면서 서버 한 대가 가장 많이 처리하는 동시 요청 수를 운영 기준으로 정합니다. 교안 01과 같은 규칙입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_results</code>의 동시 요청 수마다 목표 달성률 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_goodput / request_throughput</code>을 결과 JSON 값으로 계산합니다(반올림한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_table</code> 값이 아닙니다).</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">best_concurrency</code></strong>(int): 목표 달성률이 0.95 이상인 동시 요청 수 가운데 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_throughput</code>이 가장 큰 동시 요청 수. 세 값 모두 0.95 미만이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>으로 남고, 자가채점이 &quot;고를 수 없으니 결과 JSON을 강사에게 보여 주세요&quot;라고 알리며 멈춥니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 5-3을 참고합니다.
목표 달성률은 반올림하지 않은 결과 JSON 값으로 나눕니다.
조건을 지키고 출력 처리량이 지금까지 고른 값보다 클 때만 두 변수를 바꿉니다.
```

</details>

In [ ]:
# 지금까지 고른 동시 요청 수와 그 출력 처리량
best_concurrency = None
best_throughput = 0

# TODO (1)(2): 목표 달성률을 계산하고, 0.95 이상이면서 출력 처리량이 지금까지보다 크면 고른 값을 바꾸세요
# 동시 요청 수가 큰 순서로 봅니다
for concurrency in sorted(load_results, reverse=True):
    result = load_results[concurrency]
    attainment = ...
    if ...:
        best_concurrency = ...
        best_throughput = ...
print(f"지연 목표 TTFT {TTFT_TARGET_MS}ms, TPOT {TPOT_TARGET_MS}ms를 95% 이상 지키는 동시 요청 수: {best_concurrency} | 출력 처리량: {best_throughput:.1f} 토큰/초")

In [ ]:
# [자가채점] 고른 동시 요청 수의 목표 달성률과, 조건을 지키는 다른 동시 요청 수보다 처리량이 작지 않은지 확인합니다
# 목표 달성률 0.95 이상인 동시 요청 수(조건을 지키는 후보)
candidates = []
for concurrency in load_results:
    result = load_results[concurrency]
    if result["request_goodput"] / result["request_throughput"] >= 0.95:
        candidates.append(concurrency)
assert len(candidates) > 0, "목표 달성률이 0.95 이상인 동시 요청 수가 하나도 없어 best_concurrency가 None입니다. 이 측정으로는 고를 수 없으니 결과 JSON을 강사에게 보여 주세요."

assert best_concurrency in load_results, f"best_concurrency가 {best_concurrency!r}입니다. load_results의 키(동시 요청 수) 가운데 하나를 담으세요."
best_result = load_results[best_concurrency]
best_attainment = best_result["request_goodput"] / best_result["request_throughput"]
assert best_attainment >= 0.95, f"동시 요청 {best_concurrency}의 목표 달성률이 {best_attainment:.3f}입니다. 목표 달성률 0.95 이상인 동시 요청 수 가운데에서 고르세요."

# 조건을 지키는 다른 동시 요청 수는 고른 값보다 출력 처리량이 크지 않아야 합니다
for concurrency in candidates:
    assert load_results[concurrency]["output_throughput"] <= best_result["output_throughput"], f"동시 요청 {concurrency}도 목표 달성률 0.95 이상인데 출력 처리량이 동시 요청 {best_concurrency}보다 큽니다. 조건을 지키는 값 가운데 출력 처리량이 가장 큰 값을 고르세요."
print("✅ 통과!")

## 5. 100만 토큰당 비용

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 고른 동시 요청 수가 비용을 얼마나 줄이는지 보기 위해, 출력 토큰 100만 개를 만드는 GPU 비용을 계산합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) 시간당 요금 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_PRICE_PER_HOUR</code>는 작성 셀에 0.57달러(시간당 요금 예시)로 주어집니다. 요금은 바뀔 수 있으니 RunPod 콘솔에 표시된 내 파드의 시간당 요금으로 바꿔도 됩니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">cost_table</code></strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pd.DataFrame</code>): <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_table.copy()</code> 끝에 열 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;100만 토큰당 비용(달러)&quot;</code>를 더한 테이블. 값은 출력 처리량 열로 계산한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_PRICE_PER_HOUR / (출력 처리량 * 3600) * 1_000_000</code>이고 이 열만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.round(3)</code>으로 반올림합니다.</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">cost_ratio</code></strong>(float): <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cost_table</code>에서 동시 요청 1의 비용을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_concurrency</code>의 비용으로 나눈 값(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">동시 요청 1의 비용 / best_concurrency의 비용</code> 순서라 1 이상입니다). <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_concurrency</code>가 1이면 같은 값을 나눠 1.0이 됩니다. 측정한 동시 요청 수 가운데 지연 목표를 지키면서 비용을 줄이는 값이 없었다는 뜻입니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 6절을 참고합니다.
비용 열은 출력 처리량 열 전체에 같은 식을 한 번에 적용하고, 그 열만 반올림합니다.
비율은 동시 요청 수를 행 이름으로 둔 비용 열에서 두 값을 꺼내, 동시 요청 1의 비용을 분자에, best_concurrency의 비용을 분모에 둡니다.
```

</details>

In [ ]:
# (1) 시간당 요금(제공)
GPU_PRICE_PER_HOUR = 0.57  # 시간당 요금 예시(달러). RunPod 콘솔에 표시된 내 파드의 시간당 요금으로 바꿉니다

# TODO (2): 100만 토큰당 비용 = 시간당 요금 / (출력 처리량 * 3600) * 1_000_000
cost_table = load_table.copy()
cost_table["100만 토큰당 비용(달러)"] = ...
display(cost_table[["동시 요청 수", "출력 처리량(토큰/초)", "목표 달성률", "100만 토큰당 비용(달러)"]])

# TODO (3): 동시 요청 수를 행 이름으로 둔 비용 열에서 동시 요청 1의 비용을 best_concurrency의 비용으로 나누세요
cost_by_concurrency = cost_table.set_index("동시 요청 수")["100만 토큰당 비용(달러)"]
cost_ratio = ...
print(f"동시 요청 1의 100만 토큰당 비용은 동시 요청 {best_concurrency}의 {cost_ratio:.1f}배입니다")

In [ ]:
# [자가채점] 비용 열과 비용 식의 관계, 비용 비율을 확인합니다
expected_columns = ["동시 요청 수", "출력 처리량(토큰/초)", "요청 처리량(요청/초)", "p95 TTFT(ms)", "p95 TPOT(ms)", "goodput(요청/초)", "목표 달성률", "100만 토큰당 비용(달러)"]
assert isinstance(cost_table, pd.DataFrame), "cost_table은 load_table을 복사해 열을 더한 pd.DataFrame입니다."
assert list(cost_table.columns) == expected_columns, f"cost_table의 열이 {list(cost_table.columns)}입니다. load_table의 열 뒤에 \"100만 토큰당 비용(달러)\" 열 하나를 더하세요."
assert len(cost_table) == 3, f"cost_table이 {len(cost_table)}행입니다. 동시 요청 수 세 개의 행입니다."
cost_column = cost_table["100만 토큰당 비용(달러)"]
assert cost_column.equals(cost_column.round(3)), "\"100만 토큰당 비용(달러)\" 열을 .round(3)으로 반올림하세요."

# 비용 x 출력 처리량 x 3600 / 1,000,000은 시간당 요금으로 돌아옵니다
for concurrency, throughput, cost in zip(cost_table["동시 요청 수"], cost_table["출력 처리량(토큰/초)"], cost_table["100만 토큰당 비용(달러)"]):
    hourly = cost * throughput * 3600 / 1_000_000
    assert abs(hourly - GPU_PRICE_PER_HOUR) <= 0.02, f"동시 요청 {concurrency}: 비용 x 출력 처리량 x 3600 / 1,000,000 = {hourly:.3f}이 시간당 요금 {GPU_PRICE_PER_HOUR}와 다릅니다. 식의 3600과 1_000_000을 확인하세요."

# 비용은 출력 처리량에 반비례하므로, 두 비용의 비율은 두 출력 처리량의 비율과 거의 같습니다(best_concurrency가 1이면 1.0)
assert isinstance(cost_ratio, float), f"cost_ratio가 {cost_ratio!r}입니다. 나눈 값을 float로 담으세요."
assert cost_ratio >= 1, f"cost_ratio가 {cost_ratio:.3f}입니다. 동시 요청 1의 비용을 best_concurrency의 비용으로 나누세요(나누는 순서를 확인하세요)."
throughput_by_concurrency = cost_table.set_index("동시 요청 수")["출력 처리량(토큰/초)"]
throughput_ratio = throughput_by_concurrency[best_concurrency] / throughput_by_concurrency[1]
assert abs(cost_ratio - throughput_ratio) <= 0.05 * throughput_ratio, f"cost_ratio {cost_ratio:.2f}가 두 동시 요청 수의 출력 처리량 비율 {throughput_ratio:.2f}와 다릅니다. 동시 요청 1과 best_concurrency 행의 비용을 나눴는지 확인하세요."

print("✅ 통과!")

## 마무리

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버를 끄지 않고 교안 02로 넘어갑니다. 교안 02의 2절이 이 서버를 그대로 씁니다. 오늘 실습을 여기서 마치면 아래 접힌 블록을 따라 서버를 끕니다.</p>
</div>

<details><summary>오늘 실습을 여기서 마칠 때</summary>

**RunPod 터미널 1**(`qwen3.5-4b` 서버를 띄운 터미널)에서 **`Ctrl+C`**를 누릅니다. 명령을 입력할 수 있게 되면 `nvidia-smi`를 실행합니다.

**vLLM 프로세스가 목록에서 사라지고 GPU 메모리 사용량이 줄었으면** 서버가 꺼진 것입니다. 모델 가중치와 KV 캐시가 쓰던 메모리도 이때 반환됩니다.

</details>